# WORK9 — Clean Rerun

Fresh workspace. `work8` is archive only and is not a runtime dependency.

Business rule V0.1.2: **Known Pair + closed month + no sales row => observed zero M² (`target_available=true`)**.


# WORK9 — 01 Build Core Datasets V0.1.2

**Clean V0.1.2 rerun.** This runner forces every Supabase database session/transaction into read-only mode and writes versioned artifacts to Google Drive only when `authorization.core_dataset_build: true` in `01_config/dataset_contract_v012.yaml`.

When authorized, use **Run all**. The only secret you paste is one PostgreSQL/Supabase connection string into a hidden prompt. It is not saved to Drive.


**Important:** Work9 starts a fresh Dataset V012 lineage. No Work8 run/pointer is accepted as a runtime dependency.


In [1]:
!pip -q install 'sqlalchemy>=2.0' 'psycopg[binary]>=3.2' 'pyarrow>=16' 'pyyaml>=6'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 23.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, json, getpass, yaml, hashlib
import pandas as pd
from sqlalchemy import create_engine, text, event

ROOT = Path('/content/drive/MyDrive/work9')
CONFIG_PATH = ROOT / '01_config' / 'dataset_contract_v012.yaml'
WORK9_GATE_PATH = ROOT / '01_config' / 'work9_stage_gate_v01.yaml'
SRC_PATH = ROOT / '02_src' / 'data'
assert ROOT.exists(), f'Workspace not found: {ROOT}'
sys.path.insert(0, str(SRC_PATH))
from core_dataset_v012 import (DATASET_VERSION, build_bridge, build_dim_base, build_dim_branch, build_bravo_observation, build_fact, build_pair_panel, build_branch_panel, build_calendar, validate_core, write_parquet, write_json)

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
work9_gate = yaml.safe_load(WORK9_GATE_PATH.read_text(encoding='utf-8'))
if not work9_gate['authorization']['dataset_build']:
    raise RuntimeError('WORK9 dataset build is blocked')
if not cfg['authorization']['core_dataset_build']:
    raise RuntimeError('BUILD BLOCKED: authorization.core_dataset_build=false. Ask ChatGPT to open the build gate after spec approval.')
print('Build gate: AUTHORIZED')


Mounted at /content/drive
Build gate: AUTHORIZED


In [3]:
DB_URL = getpass.getpass('Paste Supabase PostgreSQL connection string: ')
if not DB_URL.startswith(('postgresql://', 'postgres://', 'postgresql+psycopg://')):
    raise ValueError('Expected a PostgreSQL connection string')

# Use the psycopg 3 dialect explicitly. SQLAlchemy's bare postgresql:// URL
# defaults to the psycopg2 dialect, while this notebook installs psycopg 3.
if DB_URL.startswith('postgresql://'):
    SQLALCHEMY_URL = 'postgresql+psycopg://' + DB_URL[len('postgresql://'):]
elif DB_URL.startswith('postgres://'):
    SQLALCHEMY_URL = 'postgresql+psycopg://' + DB_URL[len('postgres://'):]
else:
    SQLALCHEMY_URL = DB_URL

engine = create_engine(SQLALCHEMY_URL, pool_pre_ping=True)

# Enforce READ ONLY on every newly created DBAPI session before it enters the pool.
@event.listens_for(engine, 'connect')
def _force_session_read_only(dbapi_connection, connection_record):
    old_autocommit = dbapi_connection.autocommit
    try:
        dbapi_connection.autocommit = True
        with dbapi_connection.cursor() as cur:
            cur.execute('SET SESSION CHARACTERISTICS AS TRANSACTION READ ONLY')
    finally:
        dbapi_connection.autocommit = old_autocommit

# Verify both the session default and the current transaction mode.
with engine.connect() as conn:
    default_ro = str(conn.execute(text('SHOW default_transaction_read_only')).scalar_one()).lower()
    tx_ro = str(conn.execute(text('SHOW transaction_read_only')).scalar_one()).lower()
    if default_ro != 'on' or tx_ro != 'on':
        raise RuntimeError(
            f'Read-only verification failed: default={default_ro}, transaction={tx_ro}'
        )
    print('Supabase connection verified: READ ONLY')

DB_URL = None
SQLALCHEMY_URL = None  # do not retain the secret URL variable longer than needed


Paste Supabase PostgreSQL connection string: ··········
Supabase connection verified: READ ONLY


In [4]:
def read_table(name):
    # Defense in depth: explicitly mark each read transaction READ ONLY
    # before issuing the SELECT, even though the session default is already READ ONLY.
    with engine.connect() as conn:
        conn.execute(text('SET TRANSACTION READ ONLY'))
        tx_ro = str(conn.execute(text('SHOW transaction_read_only')).scalar_one()).lower()
        if tx_ro != 'on':
            raise RuntimeError(f'Read transaction is not READ ONLY for {name}: {tx_ro}')
        return pd.read_sql_query(text(f'SELECT * FROM {name}'), conn)

master_sku = read_table('raw.master_sku')
master_channel = read_table('raw.master_channel')
sales_monthly = read_table('raw.sales_monthly')
print({'master_sku': len(master_sku), 'master_channel': len(master_channel), 'sales_monthly': len(sales_monthly)})


{'master_sku': 15499, 'master_channel': 121, 'sales_monthly': 219412}


In [6]:
from datetime import datetime, timezone
run_id = 'core_dataset_v012_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN = ROOT / '08_runs' / run_id
AUDIT = ROOT / '06_reports' / 'audits' / run_id
AUDIT.mkdir(parents=True, exist_ok=False)
# RUN is created only after validation PASS; failed builds leave audit evidence but no dataset artifact folder.
print('run_id =', run_id)


run_id = core_dataset_v012_20260815T122509Z


In [7]:
bridge = build_bridge(master_sku)
dim_base = build_dim_base(bridge)
dim_branch = build_dim_branch(master_channel)
bravo_obs, sales_audit = build_bravo_observation(sales_monthly, bridge, dim_branch)
fact = build_fact(bravo_obs)
panel_end = pd.to_datetime(sales_monthly.loc[sales_monthly['unit'].astype(str).str.strip().str.upper().eq('M2'), 'month']).max()
print('Closed-month zero rule active through panel_end =', panel_end)
pair_panel = build_pair_panel(fact, dim_base, dim_branch, panel_end)
branch_panel = build_branch_panel(fact, dim_branch, panel_end)
calendar = build_calendar(pd.to_datetime(sales_monthly['month']).min(), panel_end)
print('Core transformations built in memory')


Closed-month zero rule active through panel_end = 2026-06-01 00:00:00
Core transformations built in memory


In [8]:
# Write review/audit evidence before final gate.
def _max_ts(df, col):
    if col not in df.columns:
        return None
    x = pd.to_datetime(df[col], errors='coerce', utc=True)
    return str(x.max()) if x.notna().any() else None

def _df_sha256(df, sort_cols):
    cols = list(df.columns)
    x = df.copy()
    usable_sort = [c for c in sort_cols if c in x.columns]
    if usable_sort:
        x = x.sort_values(usable_sort, kind='mergesort').reset_index(drop=True)
    y = x[cols].astype('string').fillna('<NULL>')
    hv = pd.util.hash_pandas_object(y, index=False).values.tobytes()
    meta = ('|'.join(cols) + '|' + '|'.join(map(str, x.dtypes))).encode('utf-8')
    return hashlib.sha256(meta + hv).hexdigest()

source_hashes = {
    'raw.master_sku': _df_sha256(master_sku, ['bravo_sku']),
    'raw.master_channel': _df_sha256(master_channel, ['branch_code']),
    'raw.sales_monthly': _df_sha256(sales_monthly, ['bravo_sku','branch_code','unit','month']),
}
source_profile = {
 'master_sku_rows': int(len(master_sku)),
 'master_channel_rows': int(len(master_channel)),
 'sales_monthly_rows': int(len(sales_monthly)),
 'sales_min_month': str(pd.to_datetime(sales_monthly['month']).min()),
 'sales_max_month': str(pd.to_datetime(sales_monthly['month']).max()),
 'sales_unit_counts': sales_monthly['unit'].astype(str).str.strip().value_counts(dropna=False).to_dict(),
 'source_asof': {
    'raw.master_sku.source_updated_at_max': _max_ts(master_sku, 'source_updated_at'),
    'raw.master_channel.loaded_at_max': _max_ts(master_channel, 'loaded_at'),
    'raw.sales_monthly.loaded_at_max': _max_ts(sales_monthly, 'loaded_at'),
 },
 'source_data_sha256': source_hashes,
}
source_schema = {
 'raw.master_sku': {c: str(t) for c,t in master_sku.dtypes.items()},
 'raw.master_channel': {c: str(t) for c,t in master_channel.dtypes.items()},
 'raw.sales_monthly': {c: str(t) for c,t in sales_monthly.dtypes.items()},
}
write_json(source_profile, AUDIT / 'source_profile.json')
write_json(source_schema, AUDIT / 'source_schema.json')
write_json({
 'bridge': len(bridge), 'dim_base': len(dim_base), 'dim_branch': len(dim_branch),
 'bravo_m2_observation': len(bravo_obs), 'fact': len(fact),
 'pair_panel': len(pair_panel), 'branch_panel': len(branch_panel), 'calendar': len(calendar),
}, AUDIT / 'row_counts.json')

key_checks = {
 'bridge_bravo_sku_duplicate_rows': int(bridge.duplicated(['bravo_sku'], keep=False).sum()),
 'dim_base_sku_duplicate_rows': int(dim_base.duplicated(['base_sku'], keep=False).sum()),
 'dim_branch_duplicate_rows': int(dim_branch.duplicated(['branch_code'], keep=False).sum()),
 'fact_duplicate_rows': int(fact.duplicated(['base_sku','branch_code','month'], keep=False).sum()),
 'bravo_observation_duplicate_rows': int(bravo_obs.duplicated(['bravo_sku','branch_code','month'], keep=False).sum()),
 'pair_panel_duplicate_rows': int(pair_panel.duplicated(['base_sku','branch_code','month'], keep=False).sum()),
 'branch_panel_duplicate_rows': int(branch_panel.duplicated(['branch_code','month'], keep=False).sum()),
 'calendar_duplicate_rows': int(calendar.duplicated(['month'], keep=False).sum()),
}
write_json(key_checks, AUDIT / 'key_uniqueness.json')

def _dup_summary(df, keys, table):
    x=df.copy()
    for c in keys:
        if c in x.columns and x[c].dtype == object:
            x[c]=x[c].astype('string').str.strip()
    if 'unit' in keys and 'unit' in x.columns:
        x['unit']=x['unit'].astype('string').str.strip().str.upper()
    dup=x.duplicated(keys, keep=False)
    return {
      'table':table, 'grain':' × '.join(keys),
      'duplicate_rows':int(dup.sum()),
      'duplicate_groups':int(x.loc[dup].groupby(keys, dropna=False).ngroups) if dup.any() else 0,
    }
pd.DataFrame([
 _dup_summary(master_sku, ['bravo_sku'], 'raw.master_sku'),
 _dup_summary(master_channel, ['branch_code'], 'raw.master_channel'),
 _dup_summary(sales_monthly, ['bravo_sku','branch_code','unit','month'], 'raw.sales_monthly'),
]).to_csv(AUDIT / 'raw_duplicate_report.csv', index=False)

excl_cols = [c for c in ['source_file','source_row_no','bravo_sku','base_sku','branch_code','month','mapping_status','branch_mapping_status','primary_exclusion_reason','secondary_flags'] if c in bravo_obs.columns]
bravo_obs.loc[bravo_obs['row_disposition'].eq('EXCLUDED'), excl_cols].to_csv(AUDIT / 'exclusion_audit.csv', index=False)

mapping_audit = bravo_obs.groupby(['row_disposition','primary_exclusion_reason','mapping_status','branch_mapping_status'], dropna=False, as_index=False).agg(
    rows=('bravo_sku','size'), gross_positive_m2=('gross_positive_m2','sum'), negative_m2=('negative_m2','sum'))
mapping_audit.to_csv(AUDIT / 'mapping_audit.csv', index=False)
structure_audit = bridge.groupby(['modeling_exclusion_reason','structure_valid','is_l1'], dropna=False, as_index=False).agg(
    bravo_rows=('bravo_sku','size'), base_skus=('base_sku','nunique'))
structure_audit.to_csv(AUDIT / 'structure_audit.csv', index=False)

stable_fields=['product_group','brand','price_group','factory_code','pull_source']
conf=[]
valid_bridge=bridge.loc[bridge['modeling_universe_valid']].copy()
for field in stable_fields:
    if field in valid_bridge.columns:
        g=valid_bridge.groupby('base_sku')[field].nunique(dropna=True)
        for base,n in g[g.gt(1)].items():
            vals=sorted(valid_bridge.loc[valid_bridge['base_sku'].eq(base),field].dropna().astype(str).unique().tolist())
            conf.append({'base_sku':base,'field':field,'distinct_nonnull':int(n),'values':' | '.join(vals)})
pd.DataFrame(conf, columns=['base_sku','field','distinct_nonnull','values']).to_csv(AUDIT / 'base_attribute_conflicts.csv', index=False)

ch = bravo_obs.loc[bravo_obs['row_disposition'].eq('MODELING_VALID') & bravo_obs['gross_positive_m2'].gt(0)].groupby('channel_relation_snapshot', dropna=False, as_index=False).agg(positive_rows=('bravo_sku','size'), gross_m2=('gross_positive_m2','sum'))
if len(ch): ch['gross_share'] = ch['gross_m2'] / ch['gross_m2'].sum()
ch.to_csv(AUDIT / 'channel_profile.csv', index=False)

rows=[]
for table_name, df, fields in [
 ('raw.master_sku', master_sku, ['bravo_sku','base_sku','master_status']),
 ('raw.master_channel', master_channel, ['branch_code','master_status']),
 ('raw.sales_monthly', sales_monthly, ['bravo_sku','branch_code','unit','month','total_quantity']),
]:
    for col in fields:
        s=df[col]
        blank = int(s.astype('string').str.strip().eq('').fillna(False).sum()) if s.dtype == object or str(s.dtype).startswith('string') else 0
        rows.append({'table':table_name,'field':col,'null_count':int(s.isna().sum()),'blank_count':blank})
pd.DataFrame(rows).to_csv(AUDIT / 'null_blank_report.csv', index=False)

eligible = bravo_obs.loc[bravo_obs['row_disposition'].eq('MODELING_VALID')]
fact_recon = {
  'positive_source_m2': float(eligible['gross_positive_m2'].sum()),
  'positive_fact_m2': float(fact['gross_positive_m2'].sum()),
  'negative_source_m2': float(eligible['negative_m2'].sum()),
  'negative_fact_m2': float(fact['negative_m2'].sum()),
}
fact_recon['positive_match'] = bool(abs(fact_recon['positive_source_m2']-fact_recon['positive_fact_m2']) <= 1e-8)
fact_recon['negative_match'] = bool(abs(fact_recon['negative_source_m2']-fact_recon['negative_fact_m2']) <= 1e-8)
write_json(fact_recon, AUDIT / 'fact_reconciliation.json')

pair_sem = {
  'rows': int(len(pair_panel)),
  'known_pairs': int(pair_panel[['base_sku','branch_code']].drop_duplicates().shape[0]),
  'actual_observed_rows': int(pair_panel['actual_observed'].sum()),
  'missing_unknown_rows': int((~pair_panel['actual_observed']).sum()),
  'negative_only_rows': int(pair_panel['actual_negative_only'].sum()),
  'negative_only_target_available_rows': int(pair_panel.loc[pair_panel['actual_negative_only'],'target_available'].sum()),
  'negative_only_nonnull_actual_target_rows': int(pair_panel.loc[pair_panel['actual_negative_only'],'actual_gross_m2'].notna().sum()),
  'future_first_positive_visibility_rows': int((pair_panel['first_positive_month_to_date'].notna() & pair_panel['first_positive_month_to_date'].gt(pair_panel['month'])).sum()),
}
write_json(pair_sem, AUDIT / 'pair_panel_validation.json')
branch_sem = {
  'rows': int(len(branch_panel)),
  'branches_with_history': int(branch_panel['branch_code'].nunique()),
  'observed_branch_months': int(branch_panel['branch_observed'].sum()),
  'missing_branch_months': int((~branch_panel['branch_observed']).sum()),
  'positive_branch_months': int(branch_panel['branch_positive'].sum()),
}
write_json(branch_sem, AUDIT / 'branch_panel_validation.json')
print('Audit evidence prepared:', AUDIT)


Audit evidence prepared: /content/drive/MyDrive/work9/06_reports/audits/core_dataset_v012_20260815T122509Z


In [9]:
validation = validate_core(sales_monthly, bravo_obs, bridge, dim_base, dim_branch, fact, pair_panel, branch_panel, calendar)
display(pd.DataFrame([{'check': k, **v} for k, v in validation['checks'].items()]))
if validation['status'] != 'PASS':
    write_json(validation, AUDIT / 'validation_summary.json')
    raise RuntimeError('VALIDATION FAIL — no dataset artifacts will be published')
print('Validation: PASS')


,check,pass,detail
0,bridge_unique,True,None
1,dim_base_unique,True,None
2,dim_branch_unique,True,None
3,fact_unique,True,None
4,pair_unique,True,None
5,branch_unique,True,None
6,calendar_unique,True,None
7,fact_base_subset_dim,True,None
8,gross_nonnegative,True,None
9,negative_nonpositive,True,None


Validation: PASS


In [10]:
# Write immutable run-scoped artifacts only after validation PASS.
RUN.mkdir(parents=True, exist_ok=False)
processed = RUN / '04_data' / 'processed'
panels = RUN / '04_data' / 'panels'
caldir = RUN / '04_data' / 'calendar'
write_parquet(bridge, processed / 'bridge_bravo_base_snapshot_v012.parquet')
write_parquet(dim_base, processed / 'dim_base_sku_snapshot_v012.parquet')
write_parquet(dim_branch, processed / 'dim_branch_snapshot_v012.parquet')
write_parquet(fact, processed / 'fact_sales_m2_monthly_v012.parquet')
write_parquet(bravo_obs, panels / 'bravo_branch_month_observation_v012.parquet')
write_parquet(pair_panel, panels / 'base_sku_branch_month_panel_v012.parquet')
write_parquet(branch_panel, panels / 'branch_month_panel_v012.parquet')
write_parquet(calendar, caldir / 'calendar_month_v012.parquet')

write_json(validation, AUDIT / 'validation_summary.json')
code_sha256 = hashlib.sha256((SRC_PATH / 'core_dataset_v012.py').read_bytes()).hexdigest()
config_sha256 = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
write_json({
 'run_id': run_id,
 'status': 'PASS',
 'dataset_version': DATASET_VERSION,
 'code_sha256': code_sha256,
 'config_sha256': config_sha256,
 'source_data_sha256': source_hashes,
 'source_asof': source_profile['source_asof'],
 'source_min_month': source_profile['sales_min_month'],
 'source_max_month': source_profile['sales_max_month'],
 'raw_rows': {'master_sku': len(master_sku), 'master_channel': len(master_channel), 'sales_monthly': len(sales_monthly)},
 'outputs': {'bridge': len(bridge), 'dim_base': len(dim_base), 'dim_branch': len(dim_branch), 'fact': len(fact), 'bravo_obs': len(bravo_obs), 'pair_panel': len(pair_panel), 'branch_panel': len(branch_panel), 'calendar': len(calendar)},
 'connection_string_saved': False
}, AUDIT / 'run_manifest.json')

print('PASS. Outputs written to:', RUN)
print('Audit written to:', AUDIT)

# Mutable Work9 pointer for the next stage. This does not alter the immutable run folder.
current_dataset = {
    'run_id': run_id,
    'status': 'PASS',
    'dataset_version': DATASET_VERSION,
    'run_dir': str(RUN),
    'audit_dir': str(AUDIT),
    'manifest_path': str(AUDIT / 'run_manifest.json'),
    'pair_panel_path': str(panels / 'base_sku_branch_month_panel_v012.parquet'),
    'branch_panel_path': str(panels / 'branch_month_panel_v012.parquet'),
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
(ROOT/'01_config'/'current_dataset_run.json').write_text(json.dumps(current_dataset, indent=2), encoding='utf-8')
print('Current dataset pointer updated:', ROOT/'01_config'/'current_dataset_run.json')


PASS. Outputs written to: /content/drive/MyDrive/work9/08_runs/core_dataset_v012_20260815T122509Z
Audit written to: /content/drive/MyDrive/work9/06_reports/audits/core_dataset_v012_20260815T122509Z
Current dataset pointer updated: /content/drive/MyDrive/work9/01_config/current_dataset_run.json


## Important
This notebook does **not** build holiday/Tết features, feature panels, train models, freeze, run Frozen Test, or publish forecasts. Those remain later gated stages.
